# D1.5 · Agent telemetry as a data source

**Function D — Security Operations → The SOC Analyst & Detection Engineer**  ·  *Security of AI*

---

**Risk.** Prompts, traces, tool calls and approvals never reach the SIEM.

**Control.** Onboard agent telemetry deliberately; decide retention.

**This lab.** Get agent telemetry into the SIEM and query it.

| | |
|---|---|
| Open-source tooling | OpenTelemetry, OpenSearch |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("D1.5"))

Agent telemetry is a data source with a property no other source has: it contains the *reasoning*, not just the action. That is useful and it is a retention problem.

In [ ]:
from cybercommons import loop, ir

trace = loop.run(loop.FakeModel(["read the config", "read the config", "patch line 12"]),
                 loop.unit_test(lambda s: s.startswith("patch"), "reached a patch"),
                 goal="fix the misconfiguration", max_steps=5)
print(trace.table())
print("\nas a telemetry record:")
print(trace.as_dict())

Three fields make this forensically useful and each has a cost: the proposals (may contain customer data), the verifier details (cheap and high value), and timing (cheap). Decide retention per field, not per record.

In [ ]:
for name, r in (("full", ir.Replay(["p"], ["result"], "glm-4.6", 0)),
                ("actions only", ir.Replay([], [], "glm-4.6", 0))):
    ok, missing = r.replayable()
    print(f"{name:14s} replayable={ok}  missing={missing}")

### Expect

The trace prints three steps ending in success, the dict form shows per-step verifier detail and timings, and the actions-only record is reported as not replayable.

### Your turn

What is your retention period for agent reasoning traces? If it is the same as for firewall logs, one of the two numbers is wrong.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/D1.5.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*